With posterior weight balancing with ctc loss and Removed residual bilstm block

  What changed
  class EnsembleModel(nn.Module):
    def __init__(self, fusion_method='add', num_classes=7):
        super(EnsembleModel, self).__init__()
        self.fusion_method = fusion_method
        input_dim = 128

        if fusion_method in ['concat', 'weighted_concat', 'attention_soft']:
            fused_dim = 256
        else:
            fused_dim = 128

        # LayerNorm for inputs
        self.norm1 = nn.LayerNorm(128)
        self.norm2 = nn.LayerNorm(128)

        if fusion_method in ['weighted_add', 'weighted_concat']:
            self.weight1 = nn.Parameter(torch.tensor(0.5))
            self.weight2 = nn.Parameter(torch.tensor(0.5))

        if fusion_method == 'attention_soft':
            self.attention = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 2),
                nn.Softmax(dim=1)
            )

        if fusion_method == 'attention_scalar':
            self.att_scalar = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 1),
                nn.Sigmoid()
            )

        # Classifier head with BatchNorm
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(fused_dim),
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, features1, features2):
        features1 = self.norm1(features1)
        features2 = self.norm2(features2)

        if self.fusion_method == 'add':
            fused = features1 + features2

        elif self.fusion_method == 'concat':
            fused = torch.cat([features1, features2], dim=1)

        elif self.fusion_method == 'weighted_add':
            weights = torch.softmax(torch.stack([self.weight1, self.weight2]), dim=0)
            fused = weights[0] * features1 + weights[1] * features2

        elif self.fusion_method == 'weighted_concat':
            weights = torch.softmax(torch.stack([self.weight1, self.weight2]), dim=0)
            w1f = weights[0] * features1
            w2f = weights[1] * features2
            fused = torch.cat([w1f, w2f], dim=1)

        elif self.fusion_method == 'attention_soft':
            concat_features = torch.cat([features1, features2], dim=1)
            attention_weights = self.attention(concat_features)  # (B, 2)
            fused = attention_weights[:, 0:1] * features1 + attention_weights[:, 1:2] * features2

        elif self.fusion_method == 'attention_scalar':
            concat_features = torch.cat([features1, features2], dim=1)  # (B, 256)
            scalar_weight = self.att_scalar(concat_features)  # (B, 1)
            fused = scalar_weight * features1 + (1 - scalar_weight) * features2

        return self.classifier(fused)


🔁 Part 1: Normalize Features Before Fusion
Why: Ensures both features1 and features2 have comparable scale — prevents one from dominating.

📍 Where: In the forward() method of your EnsembleModel class, add LayerNorm before fusion.

python
Copy
Edit
self.norm1 = nn.LayerNorm(128)
self.norm2 = nn.LayerNorm(128)
Add this normalization to the forward:

python
Copy
Edit
features1 = self.norm1(features1)
features2 = self.norm2(features2)
🎚️ Part 2: Learnable Fusion with Proper Initialization
Why: Right now, both weights are initialized as 0.5, which may not be ideal.

📍 Where: When defining self.weight1, self.weight2 in your __init__:

python
Copy
Edit
self.weight1 = nn.Parameter(torch.tensor(0.5), requires_grad=True)
self.weight2 = nn.Parameter(torch.tensor(0.5), requires_grad=True)
And ensure you use:

python
Copy
Edit
weights = torch.softmax(torch.stack([self.weight1, self.weight2]), dim=0)
This you already do — ✅ good practice.

💡 Part 3: Use Dropout + BatchNorm/LayerNorm in Attention Modules
Why: Prevents overfitting and encourages generalization in attention_scalar.

📍 Where: When defining self.att_scalar and self.attention, modify as:

python
Copy
Edit
# For attention_scalar
self.att_scalar = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 1),
    nn.Sigmoid()
)
python
Copy
Edit
# For attention_soft
self.attention = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 2),
    nn.Softmax(dim=1)
)
🔄 Part 4: Use Batch Normalization in Classifier
Why: Helps with convergence and stabilizes output from fused features.

📍 Where: Wherever you define self.classifier:

python
Copy
Edit
self.classifier = nn.Sequential(
    nn.BatchNorm1d(fused_dim),  # 128 or 256 based on fusion
    nn.Linear(fused_dim, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, num_classes)
)
Make sure fused_dim is:

128 for add, weighted_add, attention_scalar

256 for concat, weighted_concat, attention_soft

🔁 Part 5: Extend Early Stopping Patience
In your training loop (probably inside train_ensemble), modify:

python
Copy
Edit
early_stopping = EarlyStopping(patience=10, ... )  # was likely 5 before
Also reduce min_delta if it's large.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras.layers import BatchNormalization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Bidirectional, LSTM, Dropout,
                                     SeparableConv1D, ReLU, Add,
                                     GlobalAveragePooling1D, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
import random
import pickle

# Set seeds for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

In [ ]:
def load_data():
    """Load and preprocess data as in your original code"""
    # Load your data
    with open("/content/drive/MyDrive/y_train_aug.pkl", "rb") as f:
        y_train_aug = pickle.load(f)

    with open("/content/drive/MyDrive/feature_sets_dict.pkl", "rb") as f:
        feature_sets = pickle.load(f)

    # Clean data
    X_all = np.array(feature_sets["MFCC+MFDWC"])
    X_cleaned, y_cleaned = [], []

    for x, label in zip(X_all, y_train_aug):
        if label == 'calm':
            continue
        if label == 'surprised':
            label = 'surprise'
        X_cleaned.append(x)
        y_cleaned.append(label)

    X_cleaned = np.array(X_cleaned)

    # Encode labels
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.utils import to_categorical

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_cleaned)
    y_cat = to_categorical(y_encoded)

    # Split data
    X_temp, X_test, y_temp, y_test = train_test_split(X_cleaned, y_cat, test_size=0.2, stratify=y_cat.argmax(1), random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp.argmax(1), random_state=42)

    # Calculate class weights for imbalanced data
    class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print(f"Data shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")
    print(f"Class weights: {class_weight_dict}")

    return X_train, y_train, X_val, y_val, X_test, y_test, class_weight_dict, label_encoder

New ensembles and Single BiLSTM with Highway Connection and CCC - Corrected version

In [ ]:
def xception_block_1d(x, filters, kernel_size=3):
    """Xception block without residual connection as requested"""
    x = SeparableConv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = SeparableConv1D(filters, kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

def build_bilstm_xception_model(input_shape, num_classes):
    """BiLSTM + Xception model that outputs 128-dimensional features"""
    inp = Input(shape=input_shape)

    # BiLSTM
    x = Bidirectional(LSTM(128, return_sequences=True))(inp)
    x = Dropout(0.3)(x)

    # Highway connection
    input_projected = Dense(256)(inp)  # Match BiLSTM output dimension
    transform_gate = Dense(256, activation='sigmoid')(input_projected)
    carry_gate = Lambda(lambda x: 1.0 - x)(transform_gate)
    x = Add()([
        Multiply()([x, transform_gate]),
        Multiply()([input_projected, carry_gate])
    ])

    # Xception blocks (no residual as requested)
    x = xception_block_1d(x, filters=64)
    x = xception_block_1d(x, filters=128)

    # Feature extraction layer (128 dimensions)
    x = GlobalAveragePooling1D()(x)
    features = Dense(128, activation='relu', name='features')(x)
    x = Dropout(0.3)(features)

    # Final classification layer
    out = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inp, outputs=out)
    feature_model = Model(inputs=inp, outputs=features)

    return model, feature_model

In [ ]:
def train_bilstm_xception(X_train, y_train, X_val, y_val, class_weight_dict):
    """Train BiLSTM+Xception model"""
    model, feature_extractor = build_bilstm_xception_model(X_train.shape[1:], num_classes=7)

    model.compile(
        optimizer=Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True),
        ReduceLROnPlateau(patience=5, factor=0.5)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    return model, feature_extractor

In [ ]:
def mish(x):
    return x * torch.tanh(F.softplus(x))

class Mish(nn.Module):
    def forward(self, x):
        return mish(x)

class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, dim, 2).float() * -(np.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SEBlock(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(dim, dim // reduction, 1),
            nn.ReLU(),
            nn.Conv1d(dim // reduction, dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        scale = self.se(x)
        x = x * scale
        return x.transpose(1, 2)

class FeedForwardModule(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * expansion),
            Mish(),
            nn.Dropout(dropout),
            nn.Linear(dim * expansion, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return x + 0.5 * self.net(x)

class ConvolutionModule(nn.Module):
    def __init__(self, dim, kernel_size=31):
        super().__init__()
        self.pointwise1 = nn.Conv1d(dim, 2 * dim, kernel_size=1)
        self.glu = nn.GLU(dim=1)
        self.depthwise = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim)
        self.bn = nn.BatchNorm1d(dim)
        self.act = Mish()
        self.pointwise2 = nn.Conv1d(dim, dim, kernel_size=1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pointwise1(x)
        x = self.glu(x)
        x = self.depthwise(x)
        x = self.bn(x)
        x = self.act(x)
        x = self.pointwise2(x)
        return x.transpose(1, 2)

class ConformerBlock(nn.Module):
    def __init__(self, dim, heads=4, ff_expansion=4, conv_kernel=31, dropout=0.1):
        super().__init__()
        self.ff1 = FeedForwardModule(dim, ff_expansion, dropout)
        self.mha = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.conv = ConvolutionModule(dim, conv_kernel)
        self.norm2 = nn.LayerNorm(dim)
        self.ff2 = FeedForwardModule(dim, ff_expansion, dropout)
        self.norm3 = nn.LayerNorm(dim)
        self.se = SEBlock(dim)

    def forward(self, x):
        x = self.ff1(x)
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.conv(x))
        x = self.ff2(x)
        x = self.se(x)
        return self.norm3(x)

class ConformerWithCTC(nn.Module):
    def __init__(self, input_dim, model_dim=128, num_blocks=2, num_classes=7):
        super().__init__()
        self.model_dim = model_dim
        self.num_classes = num_classes

        # Input projection
        self.input_proj = nn.Linear(input_dim, model_dim)

        # Positional encoding
        self.pos_enc = PositionalEncoding(model_dim)

        # Conformer blocks
        self.conformer_blocks = nn.ModuleList([
            ConformerBlock(model_dim, heads=8, ff_expansion=4, conv_kernel=31, dropout=0.1)
            for _ in range(num_blocks)
        ])

        # Feature extraction layer (for ensemble)
        self.feature_layer = nn.Linear(model_dim, 128)

        # CTC projection layer
        self.ctc_projection = nn.Linear(model_dim, num_classes + 1)  # +1 for CTC blank

        # Classification head (for standard training)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, output_type='features'):
        # Input projection and positional encoding
        x = self.input_proj(x)
        x = self.pos_enc(x)

        # Pass through conformer blocks
        for block in self.conformer_blocks:
            x = block(x)

        if output_type == 'features':
            # For ensemble: return mean-pooled features
            pooled = x.mean(dim=1)
            return self.feature_layer(pooled)

        elif output_type == 'classification':
            # For standard classification
            pooled = x.mean(dim=1)
            features = self.feature_layer(pooled)
            return self.classifier(features)

        elif output_type == 'ctc':
            # For CTC loss: return sequence logits
            return self.ctc_projection(x)

        else:
            raise ValueError("output_type must be 'features', 'classification', or 'ctc'")

In [ ]:
def train_conformer_with_ctc(X_train, y_train, X_val, y_val, class_weights, use_ctc=True):
    """Train Conformer model with CTC loss and class weighting"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = ConformerWithCTC(input_dim=90, model_dim=128, num_blocks=2, num_classes=7).to(device)

    # Convert data to PyTorch tensors
    X_train_torch = torch.tensor(X_train, dtype=torch.float32)
    y_train_torch = torch.tensor(y_train.argmax(1), dtype=torch.long)
    X_val_torch = torch.tensor(X_val, dtype=torch.float32)
    y_val_torch = torch.tensor(y_val.argmax(1), dtype=torch.long)

    # Create data loaders
    train_loader = DataLoader(TensorDataset(X_train_torch, y_train_torch), batch_size=64, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_torch, y_val_torch), batch_size=64, shuffle=False)

    # Loss function with class weights
    weight_list = list(class_weights.values()) + [1.0]  # Add 1.0 weight for blank class
    weight_tensor = torch.tensor(weight_list, dtype=torch.float32).to(device)

    if use_ctc:
        # For CTC: use weighted CTC loss
        criterion = nn.CTCLoss(blank=7, reduction='mean')
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    else:
        # For standard classification: use weighted cross-entropy
        criterion = nn.CrossEntropyLoss(weight=weight_tensor)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    model.train()
    for epoch in range(30):
        total_loss = 0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()

            if use_ctc:
              output = model(data, output_type='ctc')  # (B, T, num_classes+1)
              input_lengths = torch.full((data.size(0),), output.size(1), dtype=torch.long)
              target_lengths = torch.ones(data.size(0), dtype=torch.long)
              target_flat = target.view(-1)
              scaled_output = output * weight_tensor.unsqueeze(0).unsqueeze(0)
              loss = criterion(scaled_output.log_softmax(2).transpose(0, 1), target_flat, input_lengths, target_lengths)


            else:
                # Standard classification training
                output = model(data, output_type='classification')
                loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            if not use_ctc:
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()
                total += target.size(0)

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)

                if use_ctc:
                    output = model(data, output_type='ctc')
                    input_lengths = torch.full((data.size(0),), output.size(1), dtype=torch.long)
                    target_lengths = torch.ones(data.size(0), dtype=torch.long)
                    target_flat = target.view(-1)
                    val_loss += criterion(output.log_softmax(2).transpose(0, 1), target_flat, input_lengths, target_lengths).item()
                else:
                    output = model(data, output_type='classification')
                    val_loss += criterion(output, target).item()
                    pred = output.argmax(dim=1)
                    val_correct += pred.eq(target).sum().item()
                    val_total += target.size(0)

        model.train()

        if epoch % 5 == 0:
            if use_ctc:
                print(f'Epoch {epoch}: Train Loss: {total_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')
            else:
                train_acc = 100. * correct / total
                val_acc = 100. * val_correct / val_total
                print(f'Epoch {epoch}: Train Loss: {total_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%')

    return model

In [ ]:
from scipy.stats import pearsonr
import numpy as np

def concordance_correlation_coefficient(y_true, y_pred):
    """Concordance Correlation Coefficient"""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    return (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2 + 1e-8)


In [ ]:
class EnsembleModel(nn.Module):
    def __init__(self, fusion_method='add', num_classes=7):
        super().__init__()
        self.fusion_method = fusion_method

        # Input dimensions
        if fusion_method in ['add', 'weighted_add', 'attention_scalar']:
            fused_dim = 128
        elif fusion_method in ['concat', 'weighted_concat', 'attention_soft']:
            fused_dim = 256

        # LayerNorm for stable input scale
        self.norm1 = nn.LayerNorm(128)
        self.norm2 = nn.LayerNorm(128)

        # Learnable weights for weighted fusion
        if fusion_method in ['weighted_add', 'weighted_concat']:
            self.weight1 = nn.Parameter(torch.tensor(0.5))
            self.weight2 = nn.Parameter(torch.tensor(0.5))

        # Attention Soft Fusion (two weights across 128-d features)
        if fusion_method == 'attention_soft':
            self.attention = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 2),
                nn.Softmax(dim=1)
            )

        # Attention Scalar Fusion (single scalar)
        elif fusion_method == 'attention_scalar':
            self.att_scalar = nn.Sequential(
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 1),
                nn.Sigmoid()
            )

        # Classifier head with normalization and dropout
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(fused_dim),
            nn.Linear(fused_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, features1, features2):
        # Normalize inputs
        features1 = self.norm1(features1)
        features2 = self.norm2(features2)

        if self.fusion_method == 'add':
            fused = features1 + features2

        elif self.fusion_method == 'concat':
            fused = torch.cat([features1, features2], dim=1)

        elif self.fusion_method == 'weighted_add':
            weights = torch.softmax(torch.stack([self.weight1, self.weight2]), dim=0)
            fused = weights[0] * features1 + weights[1] * features2

        elif self.fusion_method == 'weighted_concat':
            weights = torch.softmax(torch.stack([self.weight1, self.weight2]), dim=0)
            fused = torch.cat([weights[0] * features1, weights[1] * features2], dim=1)

        elif self.fusion_method == 'attention_soft':
            concat_features = torch.cat([features1, features2], dim=1)
            attention_weights = self.attention(concat_features)  # (B, 2)
            fused = attention_weights[:, 0:1] * features1 + attention_weights[:, 1:2] * features2

        elif self.fusion_method == 'attention_scalar':
            concat_features = torch.cat([features1, features2], dim=1)
            scalar_weight = self.att_scalar(concat_features)  # (B, 1)
            fused = scalar_weight * features1 + (1 - scalar_weight) * features2

        return self.classifier(fused)


In [ ]:
def extract_features(tf_model, pt_model, X_data, device):
    """Extract features from both models"""
    # Extract features from TensorFlow model
    tf_features = tf_model.predict(X_data, batch_size=64, verbose=0)

    # Extract features from PyTorch model
    pt_model.eval()
    X_tensor = torch.tensor(X_data, dtype=torch.float32).to(device)

    with torch.no_grad():
        pt_features = []
        for i in range(0, len(X_tensor), 64):
            batch = X_tensor[i:i+64]
            batch_features = pt_model(batch, output_type='features')
            pt_features.append(batch_features.cpu().numpy())

    pt_features = np.vstack(pt_features)
    return tf_features, pt_features

In [ ]:
def train_ensemble(tf_model, pt_model, X_train, y_train, X_val, y_val,
                  class_weights, fusion_method='add', num_epochs=50):
    """Train ensemble model"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Extracting features for ensemble training...")
    # Extract features
    train_tf_features, train_pt_features = extract_features(tf_model, pt_model, X_train, device)
    val_tf_features, val_pt_features = extract_features(tf_model, pt_model, X_val, device)

    print(f"TF features shape: {train_tf_features.shape}")
    print(f"PT features shape: {train_pt_features.shape}")

    # Create ensemble model
    ensemble = EnsembleModel(fusion_method=fusion_method, num_classes=7).to(device)

    # Convert to tensors
    train_tf_tensor = torch.tensor(train_tf_features, dtype=torch.float32).to(device)
    train_pt_tensor = torch.tensor(train_pt_features, dtype=torch.float32).to(device)
    train_y_tensor = torch.tensor(y_train.argmax(1), dtype=torch.long).to(device)

    val_tf_tensor = torch.tensor(val_tf_features, dtype=torch.float32).to(device)
    val_pt_tensor = torch.tensor(val_pt_features, dtype=torch.float32).to(device)
    val_y_tensor = torch.tensor(y_val.argmax(1), dtype=torch.long).to(device)

    # Loss and optimizer with class weights
    weight_tensor = torch.tensor(list(class_weights.values()), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = torch.optim.Adam(ensemble.parameters(), lr=1e-3)

    # Training loop
    best_val_acc = 0
    patience = 0

    for epoch in range(num_epochs):
        ensemble.train()
        train_loss = 0
        correct = 0
        total = 0

        # Mini-batch training
        batch_size = 64
        for i in range(0, len(train_tf_tensor), batch_size):
            batch_tf = train_tf_tensor[i:i+batch_size]
            batch_pt = train_pt_tensor[i:i+batch_size]
            batch_y = train_y_tensor[i:i+batch_size]

            optimizer.zero_grad()
            outputs = ensemble(batch_tf, batch_pt)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()

        # Validation
        ensemble.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for i in range(0, len(val_tf_tensor), batch_size):
                batch_tf = val_tf_tensor[i:i+batch_size]
                batch_pt = val_pt_tensor[i:i+batch_size]
                batch_y = val_y_tensor[i:i+batch_size]

                outputs = ensemble(batch_tf, batch_pt)
                loss = criterion(outputs, batch_y)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += batch_y.size(0)
                val_correct += predicted.eq(batch_y).sum().item()

        train_acc = 100. * correct / total
        val_acc = 100. * val_correct / val_total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience = 0
            torch.save(ensemble.state_dict(), f'best_ensemble_{fusion_method}.pth')
        else:
            patience += 1

        if epoch % 5 == 0:
            print(f'Epoch {epoch}: Train Loss: {train_loss/(len(train_tf_tensor)/batch_size):.4f}, '
                  f'Train Acc: {train_acc:.2f}%, Val Loss: {val_loss/(len(val_tf_tensor)/batch_size):.4f}, '
                  f'Val Acc: {val_acc:.2f}%')

        # Early stopping
        early_stopping = EarlyStopping(patience=10, min_delta=0.0003, mode='max')

    return ensemble


In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    cohen_kappa_score, matthews_corrcoef
)

def evaluate_ensemble(ensemble, tf_model, pt_model, X_test, y_test, fusion_method):
    """Evaluate ensemble model with multiple metrics"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Extract test features
    test_tf_features, test_pt_features = extract_features(tf_model, pt_model, X_test, device)

    # Convert to tensors
    test_tf_tensor = torch.tensor(test_tf_features, dtype=torch.float32).to(device)
    test_pt_tensor = torch.tensor(test_pt_features, dtype=torch.float32).to(device)

    # Predict
    ensemble.eval()
    with torch.no_grad():
      test_outputs = ensemble(test_tf_tensor, test_pt_tensor)  # 🔥 Raw logits
      probs = torch.softmax(test_outputs, dim=1)
      y_true_onehot = torch.tensor(y_test, dtype=torch.float32)

      # 🧠 CCC Calculation
      probs_np = probs.detach().cpu().numpy().flatten()
      y_true_np = y_true_onehot.detach().cpu().numpy().flatten()

      ccc_score = concordance_correlation_coefficient(y_true_np, probs_np)
      print(f"🔁 Concordance Correlation Coefficient (CCC): {ccc_score:.4f}")

      _, predicted = test_outputs.max(1)

    # True and predicted labels
    y_true = y_test.argmax(1)
    y_pred = predicted.cpu().numpy()

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n{fusion_method.upper()} Fusion Results:")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")
    print(f"Cohen's Kappa: {kappa:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "mcc": mcc,
        "kappa": kappa,
        "ccc": ccc_score
    }


In [ ]:
import os
import torch

def main():
    """Complete training pipeline"""
    # Load data
    X_train, y_train, X_val, y_val, X_test, y_test, class_weights, label_encoder = load_data()

    # Paths to save/load models
    bilstm_model_path = "/content/drive/MyDrive/bilstm_highway_connection_model.pth"
    bilstm_feat_path = "/content/drive/MyDrive/bilstm_highway_connection_feature_extractor.pth"

    # Train or Load BiLSTM+Xception model
    if os.path.exists(bilstm_model_path) and os.path.exists(bilstm_feat_path):
        print("✅ Loading saved BiLSTM+Xception model from Drive...")
        bilstm_model = torch.load(bilstm_model_path, weights_only=False)
        bilstm_feature_extractor = torch.load(bilstm_feat_path, weights_only=False)
    else:
        print("🔥 Training BiLSTM+Xception model...")
        bilstm_model, bilstm_feature_extractor = train_bilstm_xception(X_train, y_train, X_val, y_val, class_weights)

        # Save to Drive
        torch.save(bilstm_model, bilstm_model_path)
        torch.save(bilstm_feature_extractor, bilstm_feat_path)
        print("💾 Saved BiLSTM+Xception model to Drive.")

    # Train Conformer model with CTC
    conformer_model_path = "/content/drive/MyDrive/conformer_ctc_model.pth"

    if os.path.exists(conformer_model_path):
        print("✅ Loading saved Conformer model with CTC from Drive...")
        conformer_model = torch.load(conformer_model_path, weights_only=False)
    else:
        print("\n🔥 Training Conformer model with CTC...")
        conformer_model = train_conformer_with_ctc(X_train, y_train, X_val, y_val, class_weights, use_ctc=True)

        # Save to Drive
        torch.save(conformer_model, conformer_model_path)
        print("💾 Saved Conformer model with CTC to Drive.")

    # Train ensemble with different fusion methods
    fusion_methods = [
    'add', 'concat',
    'weighted_add', 'weighted_concat',
    'attention_scalar']
    results = {}

    for method in fusion_methods:
        print(f"\n🔥 Training ensemble with {method} fusion...")
        ensemble = train_ensemble(bilstm_feature_extractor, conformer_model,
                                  X_train, y_train, X_val, y_val, class_weights,
                                  fusion_method=method, num_epochs=50)

        # Evaluate ensemble
        accuracy = evaluate_ensemble(ensemble, bilstm_feature_extractor, conformer_model,
                                     X_test, y_test, fusion_method=method)
        results[method] = accuracy

    # Print final results
    print("\n📊 FINAL RESULTS:")
    print("=" * 50)
    for method, metrics in results.items():
        acc = metrics["accuracy"]
        print(f"{method.upper()} Fusion: {acc * 100:.2f}%")
    best_method = max(results.items(), key=lambda item: item[1]['accuracy'])[0]
    print(f"\nBest fusion method: {best_method.upper()}")

# Run the complete pipeline
if __name__ == "__main__":
    main()


Data shapes: Train=(13788, 40, 90), Val=(1533, 40, 90), Test=(3831, 40, 90)
Class weights: {0: np.float64(0.8730057434588385), 1: np.float64(0.8808757244043787), 2: np.float64(0.8883116883116883), 3: np.float64(0.8982271831910703), 4: np.float64(1.0014641288433381), 5: np.float64(0.9095744680851063), 6: np.float64(2.615678776290631)}
✅ Loading saved BiLSTM+Xception model from Drive...
✅ Loading saved Conformer model with CTC from Drive...

🔥 Training ensemble with add fusion...
Extracting features for ensemble training...
TF features shape: (13788, 128)
PT features shape: (13788, 128)
Epoch 0: Train Loss: 0.5852, Train Acc: 80.29%, Val Loss: 0.8713, Val Acc: 69.73%
Epoch 5: Train Loss: 0.2696, Train Acc: 89.42%, Val Loss: 1.0152, Val Acc: 70.45%
Epoch 10: Train Loss: 0.2437, Train Acc: 90.40%, Val Loss: 1.0798, Val Acc: 70.45%
Epoch 15: Train Loss: 0.2302, Train Acc: 90.88%, Val Loss: 1.1571, Val Acc: 70.52%
Epoch 20: Train Loss: 0.2195, Train Acc: 91.35%, Val Loss: 1.1911, Val Acc: 70